#  Energy Consumption Modeling for UAVs Based on Flight Data and Mission Parameters :
## 1. Data analysis and feature extraction

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.ndimage import gaussian_filter1d

In [2]:
from utils import clean_flight_df, plot_flight_altitude_speed, MAE, RMSE, MAPE

### M100 Data

In [3]:
df = pd.read_csv(r"C:\Users\raman\Desktop\Kanchan_Material\01_AirTransportAndLogistics\Course_Material\Semester_03\Research Task\12683453\flights.csv", delimiter=",", encoding="utf-8", header=0)
print(df.columns)


Index(['flight', 'time', 'wind_speed', 'wind_angle', 'battery_voltage',
       'battery_current', 'position_x', 'position_y', 'position_z',
       'orientation_x', 'orientation_y', 'orientation_z', 'orientation_w',
       'velocity_x', 'velocity_y', 'velocity_z', 'angular_x', 'angular_y',
       'angular_z', 'linear_acceleration_x', 'linear_acceleration_y',
       'linear_acceleration_z', 'speed', 'payload', 'altitude', 'date',
       'time_day', 'route'],
      dtype='object')


C:\Users\raman\AppData\Local\Temp\ipykernel_18760\3443685995.py:1: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\raman\Desktop\Kanchan_Material\01_AirTransportAndLogistics\Course_Material\Semester_03\Research Task\12683453\flights.csv", delimiter=",", encoding="utf-8", header=0)


### Data Cleaning

Sanity check

In [4]:
clean_df = clean_flight_df(df)

c:\Users\raman\Desktop\Kanchan_Material\01_AirTransportAndLogistics\Course_Material\Semester_03\Research Task\utils.py:250: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time_day'] = pd.to_datetime(df['time_day'], errors='coerce').dt.time


### Find different phase of flight using  altitude, horizontal and vertical speed. 

In [5]:
# --- your preprocessing as before ---
df = df.sort_values(["flight", "time"]).reset_index(drop=True)
df["altitude_measured"] = df["position_z"] - df.groupby("flight")["position_z"].transform("first")
df["horizontal_speed"] = np.sqrt(df["velocity_x"]**2 + df["velocity_y"]**2)
df['alt_diff'] = df.groupby('flight')['altitude_measured'].diff()
df['max_altitude_flight'] = df.groupby('flight')['altitude_measured'].transform('max')
df['min_altitude_flight'] = df.groupby('flight')['altitude_measured'].transform('min')
df["dt"] = df.groupby("flight")["time"].diff()
df["vz_from_alt_raw"] = df["alt_diff"] / df["dt"]

df["vz_from_alt"] = (
    df.groupby("flight")["vz_from_alt_raw"]
      .transform(lambda x: gaussian_filter1d(x, sigma=5, mode="nearest"))
)

df["power_w"] = df["battery_voltage"] * df["battery_current"]

# --- classify phases from vertical speed ---
def classify_phase(vz):
    if vz < -0.6:
        return "Descent (Vv < -0.6 m/s)"
    elif vz > 0.6:
        return "Climb (Vv > 0.6 m/s)"
    else:
        return "Cruise (-0.6 ≤ Vv ≤ 0.6 m/s)"

df["phase_vz"] = df["vz_from_alt"].apply(classify_phase)

# --- colored histogram ---
fig_hist = px.histogram(
    df,
    x="vz_from_alt",
    nbins=40,
    color="phase_vz",                    # color by phase
    color_discrete_map={
        "Descent (Vv < -0.6 m/s)": "red",
        "Climb (Vv > 0.6 m/s)": "green",
        "Cruise (-0.6 ≤ Vv ≤ 0.6 m/s)": "blue",
    },
    barmode="overlay",                   # overlay colors; use "relative" for stacked
    opacity=0.7,
    title="Vertical Speed Histogram by Phase",
    labels={"vz_from_alt": "Vertical Velocity V<sub>v</sub> [m/s]", "phase_vz": "Phase"},
    width=800,
    height=600,
)

# vertical lines at -0.6 and +0.6
fig_hist.add_vline(x=-0.6, line_dash="dash", line_color="black",
                   annotation_text="-0.6 (Descent limit)", annotation_position="top left")
fig_hist.add_vline(x=0.6, line_dash="dash", line_color="black",
                   annotation_text="0.6 (Climb limit)", annotation_position="top right")

# styling (same as you used)
fig_hist.update_layout(
    title_font=dict(size=28, color="black", family="open sans(body)"),
    xaxis_title_font=dict(size=22, color="black", family="open sans(body)"),
    yaxis_title_font=dict(size=22, color="black", family="open sans(body)"),
    font=dict(color="black", size=18),
)

fig_hist.show()
fig_hist.write_image("vertical_speed_histogram_by_phase.png", scale=3)





### Velocity from altitude diff and velocity_z are not same e.g. in 120
So lets do a sanity check.

In [6]:
color_sequence = ["blue", "red"]  # or your preferred colors
# Select flights to inspect
flights_to_plot = [120, 79, 279]   # add more flight numbers here if you like
sub = df[df["flight"].isin(flights_to_plot)].copy()

# Prepare data for Plotly (melt to long format for nice legends)
plot_df = sub.melt(
    id_vars=["flight", "time"],
    value_vars=["velocity_z", "vz_from_alt"],
    var_name="source",
    value_name="vz"
)

# Plot both velocities over time, faceted by Flight
fig = px.line(
    plot_df,
    x="time",
    y="vz",
    color="source",
    facet_row="flight",
    title="Raw Vertical Velocity V<sub>z</sub> vs Altitude-derived Vertical Velocity V<sub>v</sub>",
    labels={"time": "Time (s)", "vz": "Vertical Velocity (m/s)", "source": "Signal Source"},
    width=1200,
    height=350*len(flights_to_plot),
    color_discrete_sequence=color_sequence
)

fig.update_traces(line=dict(width=3))

# 👉 Darken only text + axes, keep original grid + colors
fig.update_layout(
    font=dict(color="black", size=18),
    title_font=dict(size=28, color="black"),
    legend_title_font=dict(size=20, color="black"),
    legend_font=dict(size=18, color="black"),
)

fig.update_xaxes(
    title_font=dict(size=22, color="black"),
    tickfont=dict(size=24, color="black"),
    linecolor="black",
    mirror=True
)

fig.update_yaxes(
    title_font=dict(size=22, color="black"),
    tickfont=dict(size=24, color="black"),
    linecolor="black",
    mirror=True
)

fig.write_image("velocity_plot_axes_dark.png", scale=3)
fig.show()


#### Definitions of various phases

In [7]:

# Thresholds
NEARLY_ZERO_HORIZONTAL_VEL   = 0.3   # m/s 
NEARLY_ZERO_VERTICAL_VEL     = 0.2   # m/s

CRUISE_MAX_VERTICAL_VEL    = 0.6   # m/s
CRUISE_MIN_VERTICAL_VEL    = -0.6  # m/s

TAXI_ALTITUDE_MAX          = 1.0   # m   # max altitude to be considered taxiing
NEARLY_ZERO_POWER_W        = 20.0  # W   # max power to be considered taxiing

# Start with a default phase
df['phase'] = 'other'

# --- Taxi: low altitude, nearly no motion ---
taxi_mask = (
    (df['horizontal_speed'].abs() < NEARLY_ZERO_HORIZONTAL_VEL) &
    (df['vz_from_alt'].abs()      < NEARLY_ZERO_VERTICAL_VEL) &
    (df['power_w']      < NEARLY_ZERO_POWER_W)
)
df.loc[(df['phase'] == 'other') & taxi_mask, 'phase'] = 'taxi'


# --- Hover: nearly no motion ---
# 1. horizontal speed below hover max
# 2. vertical speed below hover max
hover_mask = (
    (df['horizontal_speed'].abs() < NEARLY_ZERO_HORIZONTAL_VEL) &
    (df['vz_from_alt'].abs()      < NEARLY_ZERO_VERTICAL_VEL) 
)
df.loc[(df['phase'] == 'other') & hover_mask, 'phase'] = 'hover'

# --- Cruise: nearly no vertical motion, high altitude ---
# 1. vertical speed below cruise max
# 2. altitude above 80% of max altitude in flight
cruise_mask = (
    (df['vz_from_alt'] < CRUISE_MAX_VERTICAL_VEL) &
    (df['vz_from_alt'] > CRUISE_MIN_VERTICAL_VEL) &
    (df['altitude_measured'] > 0.8 * df['max_altitude_flight'])
)
df.loc[(df['phase'] == 'other') & cruise_mask, 'phase'] = 'cruise'

# ---- CLIMB ----
# 1. vertical speed is greater than climb vertical speed minimum
climb_mask = (df['vz_from_alt'] > CRUISE_MAX_VERTICAL_VEL)
df.loc[(df['phase'] == 'other') & climb_mask, 'phase'] = 'climb'

# ---- DESCENT ----
# 1. vertical speed is less than negative of climb vertical speed minimum
descent_mask = (df['vz_from_alt'] < CRUISE_MIN_VERTICAL_VEL)
df.loc[(df['phase'] == 'other') & descent_mask, 'phase'] = 'descent'




### Find percentage of other phase in data

In [8]:
other_count = (df['phase'] == 'other').sum()
total_count = len(df)

percentage_other = (other_count / total_count) * 100
print(f"Other data percentage: {percentage_other:.2f}%")



Other data percentage: 3.97%


### Visualize different phase of flight with Line graph

In [9]:
flight_id = 79

d = df[df["flight"] == flight_id].copy()

# ------------------ Define colors for phases ------------------
phase_colors = {
    "climb":   "rgba(0,255,0,0.35)",      # light green
    "descent": "rgba(255,0,0,0.35)",      # light red
    "hover":   "rgba(255,165,0,0.35)",    # light orange
    "cruise":  "rgba(0,0,255,0.35)",      # light blue
    "taxi":    "rgba(255,255,0,0.35)",    # light yellow
    "other":   "rgba(150,150,150,0.35)"   # light gray
}

# ------------------ Build customdata for hover ------------------
customdata = np.stack((
    d["altitude_measured"],
    d["vz_from_alt"],
    d["horizontal_speed"],
    d["phase"]
), axis=-1)

fig = plot_flight_altitude_speed(
    d,
    customdata=customdata,
    phase_colors=phase_colors,
    flight_id=flight_id,
    show=True
)

# Layout for size, fonts, margins
fig.update_layout(
    width=2000,
    height=1000,
    font=dict(size=28),
    title_font=dict(size=34),
    legend=dict(font=dict(size=32),itemsizing='constant'),
    margin=dict(l=100, r=100, t=120, b=100)
)

# Darker text for title, axes, legend


# 👉 Make axes lines and tick labels dark, without changing background/grid
fig.update_xaxes(
    title_font=dict(size=32, color="black"),
    tickfont=dict(size=28, color="black"),
    linecolor="black",
    mirror=True
)

fig.update_yaxes(
    title_font=dict(size=32, color="black"),
    tickfont=dict(size=28, color="black"),
    linecolor="black",
    mirror=True
)
for tr in fig.data:
    if isinstance(tr.name, str) and tr.name.startswith("Phase:"):
        tr.showlegend = False

# 2) Define opaque colors just for the legend
legend_phase_colors = {
    "climb":   "rgba(0,255,0,1)",
    "descent": "rgba(255,0,0,1)",
    "hover":   "rgba(255,165,0,1)",
    "cruise":  "rgba(0,0,255,1)",
    "taxi":    "rgba(255,255,0,1)",
    "other":   "rgba(150,150,150,1)",
}

# 3) Add dummy traces that only appear in the legend
for phase, color in legend_phase_colors.items():
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None],          # nothing drawn on the graph
            mode="lines",
            line=dict(color=color, width=8),
            name=f"Phase: {phase}",
            showlegend=True,
        )
    )

# Export for PPT
fig.write_image("phase_segmentation_79_1.svg")
# or:
# fig.write_image("phase_segmentation_79.png", scale=3)
fig.show()



### Task two

In [10]:
def compute_wind_vector(df):
    rad = np.deg2rad(df["wind_angle"])
    wind_x = -df["wind_speed"] * np.cos(rad)
    wind_y = -df["wind_speed"] * np.sin(rad)
    return wind_x, wind_y

def compute_airspeed(df, wind_x, wind_y):
    return np.sqrt(
        (df["velocity_x"] - wind_x)**2 +
        (df["velocity_y"] - wind_y)**2
    )

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [12]:
fig_hist = px.histogram(
    df,
    x="payload",
    nbins=10,
    title="Payload Histogram",
    labels={"payload_kg": "Payload [kg]"}
)

fig_hist.show()


In [13]:
P_real     = df["power_w"].values

# wind speed calculations
wind_x, wind_y = compute_wind_vector(df)
df["wind_x"] = wind_x
df["wind_y"] = wind_y
df["airspeed"] = compute_airspeed(df, wind_x, wind_y)

V_air      = df["airspeed"].values
Vz         = df["vz_from_alt"].values
segment    = df["phase"].values
payload_kg = df["payload"].values / 1000.0

# ------------------------------------------------------
# 2. Physical Parameters for DJI M100
# ------------------------------------------------------
g   = 9.81
rho = 1.225

M0   = 3.68                      # empty mass [kg]
mass = M0 + payload_kg           # total mass [kg]
W    = mass * g                  # weight [N]

# Rotor geometry (13-inch props)
R_prop = 0.165                   # rotor radius [m]
A_rot  = np.pi * R_prop**2       # single rotor disk area [m^2]
A_tot  = 4 * A_rot               # total rotor disk area [m^2]

# Reference area & drag coefficient (from literature)
RAD   = 0.43                     # rotor axis distance [m]
# S_ref = np.pi * (RAD / 2.0)**2   # reference area [m^2]
Cd    = 0.15                    # from external UAS paper
S_ref = 0.05           # S_ref = W*H*k = 0.65*0.22*0.35 width and height are from M100 mannual
seg = segment  # shorthand

# ------------------------------------------------------
# 3. Induced Velocity & Hover Power (Momentum Theory)
# ------------------------------------------------------
vh = np.sqrt(W / (2.0 * rho * A_tot))   # induced velocity [m/s]

P_hover_base = (W ** 1.5) / np.sqrt(2.0 * rho * A_tot)  # hover induced power

# ------------------------------------------------------
# 4. Compute Base Mechanical Power (Segment-wise)
# ------------------------------------------------------
P_mech_base = np.zeros_like(P_real, dtype=float)

# --- Hover ---
mask_hover = (seg == "hover")
P_mech_base[mask_hover] = P_hover_base[mask_hover]

# --- Climb ---
mask_climb = (seg == "climb")
if mask_climb.any():
    RoC   = Vz[mask_climb]
    vh_c  = vh[mask_climb]
    ratio = RoC / (2.0 * vh_c)
    P_mech_base[mask_climb] = P_hover_base[mask_climb] * (
        ratio + np.sqrt(ratio**2 + 1.0)
    )

# --- Descent ---
mask_descent = (seg == "descent")
if mask_descent.any():
    RoD   = Vz[mask_descent]
    vh_d  = vh[mask_descent]
    ratio_d = RoD / (2.0 * vh_d)
    ratio_d = np.clip(ratio_d, -5.0, 5.0)
    P_desc = P_hover_base[mask_descent] * (
        ratio_d + np.sqrt(ratio_d**2 + 1.0)
    )
    P_mech_base[mask_descent] = np.maximum(P_desc, 0.3 * P_hover_base[mask_descent])

# --- Cruise ---
mask_cruise = (seg == "cruise")
if mask_cruise.any():
    Vc = V_air[mask_cruise]
    P_parasite = 0.5 * rho * Cd * S_ref * (Vc**3)
    P_mech_base[mask_cruise] = P_hover_base[mask_cruise] + P_parasite

# Transition = ignored
mask_transition = (seg == "other")


# ------------------------------------------------------
# 5. Estimate Effective Efficiency η per Segment
# ------------------------------------------------------
eta_seg = {}

for s in ["hover", "climb", "descent", "cruise"]:
    mask_s = (seg == s) & (P_real > 30) & (P_mech_base > 0)
    if mask_s.sum() < 50:
        continue

    Pm = P_mech_base[mask_s]
    Pr = P_real[mask_s]

    # eta = Σ Pm² / Σ (Pm * Pr)
    eta_hat = np.sum(Pm**2) / np.sum(Pm * Pr)
    eta_seg[s] = eta_hat

print("Estimated segment-wise efficiencies (mechanical → battery):")
for s, eta_hat in eta_seg.items():
    print(f"  {s:8s}: eta_seg = {eta_hat:.3f}")


# ------------------------------------------------------
# 6. Build Final Battery Power Model
# ------------------------------------------------------
P_model_phys = np.full_like(P_real, np.nan, dtype=float)

for s in ["hover", "climb", "descent", "cruise"]:
    mask_s = (seg == s) & (P_mech_base > 0)
    if s not in eta_seg:
        continue
    P_model_phys[mask_s] = P_mech_base[mask_s] / eta_seg[s]

df["P_model_phys"] = P_model_phys


# ------------------------------------------------------
# 7. Evaluate Power Model Accuracy
# ------------------------------------------------------
valid_eval = (P_real > 30) & (seg != "transition") & ~np.isnan(P_model_phys)

MAE  = mean_absolute_error(P_real[valid_eval], P_model_phys[valid_eval])
RMSE = np.sqrt(mean_squared_error(P_real[valid_eval], P_model_phys[valid_eval]))
MAPE = (np.abs(P_model_phys[valid_eval] - P_real[valid_eval]) /
        P_real[valid_eval]).mean() * 100.0

print("\n===== FINAL PHYSICS MODEL RESULTS =====")
print(f"MAE  = {MAE:.2f} W")
print(f"RMSE = {RMSE:.2f} W")
print(f"MAPE = {MAPE:.2f} %")


print("\n===== ERROR BY SEGMENT (NO TRANSITION) =====")
for s in ["hover", "climb", "descent", "cruise"]:
    mask_s = (seg == s) & (P_real > 30) & ~np.isnan(P_model_phys)
    if mask_s.sum() < 50:
        continue
    mae_s  = mean_absolute_error(P_real[mask_s], P_model_phys[mask_s])
    rmse_s = np.sqrt(mean_squared_error(P_real[mask_s], P_model_phys[mask_s]))
    mape_s = np.mean(np.abs(P_model_phys[mask_s] - P_real[mask_s]) / P_real[mask_s]) * 100
    print(f"{s:10s} | N={mask_s.sum():6d} | MAE={mae_s:8.2f} | RMSE={rmse_s:8.2f} | MAPE={mape_s:8.2f}%")
    #print(f"{s:10s} | N={mask_s.sum():6d} | MAE={mae_s:8.2f} | RMSE={rmse_s:8.2f} | MAPE={mape_s:6.2f}%")


# ------------------------------------------------------
# 8. Save Output
# ------------------------------------------------------
out_name = "M100_task2_physics_segment_eff.csv"
df.to_csv(out_name, index=False)
print(f"\nSaved: {out_name}")
print(np.unique(P_hover_base.round(2)))


Estimated segment-wise efficiencies (mechanical → battery):
  hover   : eta_seg = 0.829
  climb   : eta_seg = 0.534
  descent : eta_seg = 0.462
  cruise  : eta_seg = 0.523

===== FINAL PHYSICS MODEL RESULTS =====
MAE  = 62.35 W
RMSE = 88.70 W
MAPE = 24.76 %

===== ERROR BY SEGMENT (NO TRANSITION) =====
hover      | N= 12474 | MAE=  201.86 | RMSE=  211.64 | MAPE=  231.04%
climb      | N= 29949 | MAE=   57.04 | RMSE=   82.64 | MAPE=   10.49%
descent    | N= 48264 | MAE=   45.77 | RMSE=   64.36 | MAPE=    9.60%
cruise     | N=103112 | MAE=   54.78 | RMSE=   73.79 | MAPE=   11.04%

Saved: M100_task2_physics_segment_eff.csv
[236.92 261.47 286.81 312.92]


In [ ]:


# ------------------------------------------------------
# 9. Prepare data for Plotly
# ------------------------------------------------------

# Make sure the modeled power is in the DataFrame
# (P_model_phys is the numpy array you computed above)
df["P_model_phys"] = P_model_phys

# Optional: filter to valid samples (same logic as your evaluation)
valid_plot = (df["power_w"] > 30) & ~np.isnan(df["P_model_phys"])
df_plot = df.loc[valid_plot].copy().reset_index(drop=True)

# Use sample index as x-axis (replace with time if you have it)
df_plot["sample_idx"] = np.arange(len(df_plot))

# Build long format for real vs model
df_long = pd.DataFrame({
    "sample_idx": np.concatenate([df_plot["sample_idx"].values,
                                  df_plot["sample_idx"].values]),
    "phase":      np.concatenate([df_plot["phase"].values,
                                  df_plot["phase"].values]),
    "source":     np.concatenate([
                        np.full(len(df_plot), "P_real"),
                        np.full(len(df_plot), "P_model_phys")
                   ]),
    "power_W":    np.concatenate([df_plot["power_w"].values,
                                  df_plot["P_model_phys"].values])
})

phase_order = ["hover", "climb", "descent", "cruise"]

# ------------------------------------------------------
# 10. Time-series: real vs model, faceted by phase
# ------------------------------------------------------
fig_time = px.line(
    df_long,
    x="sample_idx",
    y="power_W",
    color="source",
    facet_row="phase",
    category_orders={"phase": phase_order},
    labels={
        "sample_idx": "Sample index",
        "power_W": "Power [W]",
        "source": "Power type"
    },
    title="Real vs Modeled Power by Phase"
)
 #Darker legend + larger font
fig_time.update_layout(
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.05,
        font=dict(size=24, color="black")  # <-- darker legend text
    ),
    title_font=dict(size=32, color="black"),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

# Darker axes & remove grid
fig_time.update_xaxes(
    showgrid=True,  # <-- remove vertical grid
    linecolor="black",
    tickfont=dict(color="black", size=24),
    title_font=dict(color="black", size=28, family="Open Sans")
)
fig_time.update_yaxes(
    showgrid=True,  # <-- remove horizontal grid
    linecolor="black",
    tickfont=dict(color="black", size=24),
    title_font=dict(color="black", size=28, family="Open Sans")
)
fig_time.for_each_annotation(
    lambda a: a.update(
        text=a.text.split("=")[-1],
        font=dict(color="black", size=24, family="Open Sans")  # <-- darker & bigger
    )
)







fig_time.update_layout(height=900, width=1400)   # increase width (adjust value as needed)

# clean facet labels ("phase=hover" -> "hover")
fig_time.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_time.show()
fig_time.write_image("Real_vs model_power_1.svg")





In [15]:


# ------------------------------------------------------
# TABLE: Mechanical Power Without Efficiency by Segment
# ------------------------------------------------------

power_table = []

for s in ["hover", "climb", "descent", "cruise"]:
    mask_s = (seg == s) & (P_mech_base > 0)
    if mask_s.any():
        power_table.append({
            "Segment": s,
            "Mean Power (W)": np.mean(P_mech_base[mask_s]),
            "Median Power (W)": np.median(P_mech_base[mask_s]),
            "Min Power (W)": np.min(P_mech_base[mask_s]),
            "Max Power (W)": np.max(P_mech_base[mask_s]),
            "Samples": mask_s.sum()
        })

power_table_df = pd.DataFrame(power_table)
print("\n=== Mechanical Power Table (NO Efficiency Applied) ===")
print(power_table_df.to_string(index=False))



=== Mechanical Power Table (NO Efficiency Applied) ===
Segment  Mean Power (W)  Median Power (W)  Min Power (W)  Max Power (W)  Samples
  hover      252.071353        236.920871     236.920871     312.922782    12722
  climb      321.798926        325.889386     248.014408     405.357933    29954
descent      231.111366        229.217769     170.799391     300.046850    48272
 cruise      268.215951        264.613797     236.920871     366.315583   103112


### Task 3

In [16]:
allowed_phases = ["cruise", "descent", "climb", "hover"]

df["phase"] = df["phase"].str.lower().str.strip()   # normalize

df_filtered = df[df["phase"].isin(allowed_phases)]

In [17]:
import math
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


# ========= 1. Quaternion → roll, pitch, yaw =========
# Quaternion assumed in (x, y, z, w) convention

def quat_to_euler(x, y, z, w):
    # Normalize to avoid numerical issues
    norm = math.sqrt(x*x + y*y + z*z + w*w)
    if norm == 0:
        return 0.0, 0.0, 0.0
    x /= norm
    y /= norm
    z /= norm
    w /= norm

    # roll (x-axis rotation)
    sinr_cosp = 2.0 * (w * x + y * z)
    cosr_cosp = 1.0 - 2.0 * (x * x + y * y)
    roll = math.atan2(sinr_cosp, cosr_cosp)

    # pitch (y-axis rotation)
    sinp = 2.0 * (w * y - z * x)
    if abs(sinp) >= 1:
        # use 90 degrees if out of range
        pitch = math.copysign(math.pi / 2.0, sinp)
    else:
        pitch = math.asin(sinp)

    # yaw (z-axis rotation)
    siny_cosp = 2.0 * (w * z + x * y)
    cosy_cosp = 1.0 - 2.0 * (y * y + z * z)
    yaw = math.atan2(siny_cosp, cosy_cosp)

    return roll, pitch, yaw

# Apply to the whole dataframe
rpy = df_filtered.apply(
    lambda row: pd.Series(
        quat_to_euler(
            row["orientation_x"],
            row["orientation_y"],
            row["orientation_z"],
            row["orientation_w"],
        ),
        index=["roll", "pitch", "yaw"],
    ),
    axis=1,
)

df_filtered = pd.concat([df_filtered, rpy], axis=1)

In [18]:
df_filtered["phase"] = df_filtered["phase"].str.lower().str.strip()

df_filtered["is_cruise"]  = df_filtered["phase"] == "cruise"
df_filtered["is_climb"]   = df_filtered["phase"] == "climb"
df_filtered["is_descent"] = df_filtered["phase"] == "descent"
df_filtered["is_hover"]   = df_filtered["phase"] == "hover"

In [19]:
# ========= 2. Define features & target =========
target_col = "power_w"

feature_cols = ['wind_speed', 
                'wind_angle', 
                'roll', 
                'pitch', 
                'yaw', 
                'velocity_x', 
                'velocity_y', 
                'velocity_z',
                'vz_from_alt', 
                'angular_x', 
                'angular_y', 
                'angular_z', 
                'linear_acceleration_x', 
                'linear_acceleration_y', 
                'linear_acceleration_z',
                'is_cruise',
                'is_climb',
                'is_descent',
                'is_hover']

X = df_filtered[feature_cols]
y = df_filtered[target_col]


# ========= 3. Train / test split =========
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
# ========= 4. Train Random Forest model =========
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [21]:
# ========= 6. Evaluate =========
y_pred = rf.predict(X_test)
rmse = math.sqrt(mean_squared_error(y_test, y_pred))

print("Test RMSE:", rmse)

# Optional: see feature importances
importances = pd.Series(rf.feature_importances_, index=feature_cols)
print(importances.sort_values(ascending=False))


Test RMSE: 50.37834517387928
is_hover                 0.237296
velocity_z               0.172280
wind_speed               0.128811
linear_acceleration_z    0.111990
yaw                      0.065878
roll                     0.047690
vz_from_alt              0.038243
velocity_y               0.029576
wind_angle               0.025529
velocity_x               0.025068
pitch                    0.023133
angular_z                0.021479
linear_acceleration_y    0.019818
linear_acceleration_x    0.018756
angular_y                0.017017
angular_x                0.016425
is_cruise                0.000623
is_climb                 0.000207
is_descent               0.000182
dtype: float64


In [22]:
MAE  = mean_absolute_error(y_test, y_pred)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
MAPE = (np.abs(y_pred - y_test) /
        y_test).mean() * 100.0

print("\n===== FINAL RANDOM FOREST MODEL RESULTS =====")
print(f"MAE  = {MAE:.2f} W")
print(f"RMSE = {RMSE:.2f} W")
print(f"MAPE = {MAPE:.2f} %")

print("\n===== ERROR BY SEGMENT (NO TRANSITION) =====")
for s in ["hover", "climb", "descent", "cruise"]:
    mask_s = X_test["is_" + s]
    mae_s  = mean_absolute_error(y_test[mask_s], y_pred[mask_s])
    rmse_s = np.sqrt(mean_squared_error(y_test[mask_s], y_pred[mask_s]))
    print(f"{s:10s} | N={mask_s.sum():6d} | MAE={mae_s:8.2f} | RMSE={rmse_s:8.2f}")



===== FINAL RANDOM FOREST MODEL RESULTS =====
MAE  = 35.87 W
RMSE = 50.38 W
MAPE = 9.23 %

===== ERROR BY SEGMENT (NO TRANSITION) =====
hover      | N=  2556 | MAE=   40.08 | RMSE=   72.40
climb      | N=  6050 | MAE=   33.97 | RMSE=   47.81
descent    | N=  9682 | MAE=   33.26 | RMSE=   46.47
cruise     | N= 20524 | MAE=   37.14 | RMSE=   49.54
